# GRAPE on a single spin

This in an introductory example to the [GRAPE](10.1016/j.jmr.2004.11.004) method and its implementation in this software package.

#### 1. Generate a PWC pulse shape

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from cthree.quantity import Quantity

from cthree.signal.envelopes import GaussEnvelope
from cthree.signal.pwc_generator import PWCGenerator

First, let's generate a piecewise constant (PWC) pulse envelope for the Gaussian pulse. For GRAPE, we need to specify an anstanz for piecewise constant controls at a given resolution. Here, we setup a Gaussian initial guess, sampling at 21 points during a gate time of 20ns.

In [ ]:
t_final = 20e-9
tlist = np.linspace(0, t_final, 21)
tone = GaussEnvelope(amplitude=Quantity(np.pi / t_final / 3, -np.pi / t_final, np.pi / t_final))
tone.t_final.setValue(t_final)
gen = PWCGenerator(envelopes=[tone], tlist=tlist)
gen.multiplyFlatTop = True

We have added the option `multiplyFlatTop`, to ensure the pulse to start and end smoothly at 0 and `t_final`.

In [ ]:
ts = np.linspace(0, t_final, 501)

plt.plot(ts / 1e-9, tone.computeOutput(ts) / 1e6, label="Smooth curve")
plt.plot(ts / 1e-9, np.real(gen.generateSignal(ts)) / 1e6, ls="--", label="Inphase")
plt.plot(
    ts / 1e-9,
    np.imag(gen.generateSignal(ts)) / 1e6,
    ls="--",
    label="Quadrature",
)

plt.xlabel("Time [in ns]")
plt.ylabel("Amplitude [in MHz]")
plt.legend()

#### 2. Define Hamiltonian in the rotating frame of drive

As a simple toy model, we use a single spin.

In [ ]:
from cthree.model.closed_system import ClosedSystem
from cthree.model.rotating_frame_drive import RotatingFrameDrive
from cthree.model.hamiltonian import Hamiltonian


class SpinRWA(Hamiltonian):
    """A Single Spin."""

    def __init__(self, drives=None):
        super().__init__(drives)
        self.sigma_p = np.array([[0j, 1], [0, 0]])
        self.dim = 2

    def getMatrixOneTime(self, t):
        """Just sigma-X."""
        return self._drives[0].get_matrix_one_time(self.sigma_p, t)

    def gradient(self, t):
        """Gradient is just the drive matrix."""
        return self._drives[0].gradient(self.sigma_p, t)


drive = RotatingFrameDrive(gen)
spin = SpinRWA(drives=[drive])
model = ClosedSystem(spin)

In [ ]:
from cthree.measurement.state_transfer_fidelity import StateTransferFidelityGRAPE
from cthree.propagation.scipy_expm_grape import ScipyExpmGRAPE


prop = ScipyExpmGRAPE(model, res=1e9)

init = np.array([[1.0], [0]])  # |0>
target = np.array([[0.0], [1]])  # |1>

prop.setInitialState(init)
prop.setTargetState(target)

zeroone = StateTransferFidelityGRAPE(
    propagation=prop,
    initialState=init,
    targetState=target,
    times=tlist,
)

In [ ]:
def plotStates():
    """Plot the states."""
    ts = np.linspace(0, t_final, 1001)
    states = prop.propagate(ts)
    sig = gen.generateSignal(ts)

    fig, ax = plt.subplots(2, figsize=(4, 4), sharex=True)
    ax[0].plot(ts / 1e-9, np.real(sig), label="I")
    ax[0].plot(ts / 1e-9, np.imag(sig), label="Q")
    ax[0].legend(loc=1)
    ax[0].set_ylabel("Field [MHz]")
    ax[1].plot(ts / 1e-9, np.abs(states)[:, :, 0] ** 2)
    ax[1].set_ylabel("Population")
    ax[-1].set_xlabel("Time [ns]")
    return fig, ax


plotStates()

In [ ]:
from cthree.optimisation_map import OptimisationMap
from cthree.optimisers.scipy_optimiser_gradient import ScipyOptimiserGradient


optmap = OptimisationMap()
optmap.add(gen)
optmap.registerParamsWithOptimisables()

optGrad = ScipyOptimiserGradient(zeroone, optimisables=optmap)

In [ ]:
optGrad.optimise()

For this simple example, we reach near perfect fidelity within a few iterations.

In [ ]:
plotStates()